In [ ]:
# Scenario: Smart Factory Temperature Monitoring
# A factory monitors the daily average temperature (°C) of a critical machine.
# The goal is to predict tomorrow’s temperature using the last 5 days of 
# temperature readings in order to:
# Detect overheating trends
# Schedule preventive maintenance
# Avoid machine failure

In [1]:

import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense


In [2]:
temps = np.array([68, 70, 71, 73, 74, 76, 78, 80, 82, 85], dtype=float)

In [3]:
scaler = MinMaxScaler()
temps_scaled = scaler.fit_transform(temps.reshape(-1, 1))

In [5]:
SEQ_IN = 5   # use last 5 days
X, y = [], []

for i in range(len(temps_scaled) - SEQ_IN):
    X.append(temps_scaled[i:i+SEQ_IN])
    y.append(temps_scaled[i+SEQ_IN])

X = np.array(X)
y = np.array(y)

In [6]:
# Reshape for RNN: (samples, time_steps, features)
X = X.reshape((X.shape[0], SEQ_IN, 1))

In [7]:
model=Sequential([
    SimpleRNN(32,activation='tanh',input_shape=(SEQ_IN,1)),
    Dense(1)
])

model.compile(optimizer='adam',loss='mse')
model.summary()

c:\Users\Akash.Aswar\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,121 (4.38 KB)

 Trainable params: 1,121 (4.38 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.fit(X,y,epochs=200,verbose=0)

In [9]:
last5days=temps_scaled[-SEQ_IN:]
input_seq=last5days.reshape(1,SEQ_IN,1)

next_temp_scaled=model.predict(input_seq,verbose=0)[0,0]
next_temp=scaler.inverse_transform([[next_temp_scaled]])[0,0]

In [10]:
print("Last 5 days temperatures : ",temps[-5:])
print(f"\nPredicted next temperture(SimpleRNN): {next_temp} celcius")

Last 5 days temperatures :  [76. 78. 80. 82. 85.]

Predicted next temperture(SimpleRNN): 85.79244089126587 celcius
